<h1 style = "color : #7328baff; text-align : center;"><em>Where should I live?</em> - Data Science in Action Notebook</h1>
<p style = "font-size : 16px; text-align: center;">
In this final part of the project, we will bring the data we have cleaned and complemented (with webscrapped coordinates) integrating them with more tools and processes transforming it into <b>meaningful insights or tools</b>.</p>

<br>
<p style = "font-size : 12px; text-align: center;"><b>NOVA IMS</b></p>
<p style = "font-size : 10px; text-align: center;">Programming for Data Science</p>
<p style = "font-size : 10px; text-align: center;">Diogo Gonçalves, João Marques, Juan Mendes & Gustavo Franco</p>
<br>

<h2  style = "color : #7328baff;"> Tools and Structure</h2>

<p>The tools we are giving are the follwoing:</p>
<ul><li>A dashboard to compare 2 different cities to help you choose where to live;</li>
<li>A dashboard where you can rank the variables most important for your lifestyle and preferences, suggesting you a city to live;</li>
<li>A map showing you, based on your choice of city to live, the holiday destinations you have around.</li></ul>

<p>To accomplish this, we designed this notebook that is separate into 3 parts of implementation:</p>
<ul><li><b>Web Scrapping</b> - where we used <b>Requests</b> to go to get more information on each city from <code>https://www.numbeo.com/</code></li>
<li><b>Dashboard Designing and Engeneering</b> - where we used <b>Panel</b> to build an interactive Dashboard featuring a page to compare 2 different cities and a page where you can tell what are your main concerns and it will suggest you a city to live</li>
<li><b>Map with Holiday Destinations</b> - using <b>Geopandas</b> we have integrated a dataset with European holiday destinations, and it will suggest you the destinations you have around the city you chose/are choosing to live. We believe that more than <i>living</i> nowadays it is also very important to have good quality holidays.</li></ul>

<hr style = "border: 3px solid #7328baff;">
<h2  style = "color : #7328baff;"> Imports & Dataset Loadings</h2>

In [ ]:
#!pip install panel
#!pip install geopandas

In [ ]:
from bs4 import BeautifulSoup
import requests
import re    
import pandas as pd
import numpy as np
import panel as pn
import plotly.graph_objects as go
import geopandas as gpd

import warnings

warnings.filterwarnings('ignore')

In [27]:
city_data = pd.read_csv("city_data_clean.csv")
geo_city_data = pd.read_csv('geo_city_data.csv')

<hr style = "border: 3px solid #7328baff;">
<h2  style = "color : #7328baff;">Web Scraping</h2>

<p>Starting off by getting the links for each part of the Numbeo website - we chose here to go using <code>Requests</code> packadge as we had already used <code>Selenium</code> on this project</p>

In [28]:
url_dictionary = {"Food Prices": "https://www.numbeo.com/food-prices/in/city",
"Gas Prices Calculator" : "https://www.numbeo.com/gas-prices/in/city",
"Salary Calculator" : "https://www.numbeo.com/cost-of-living/prices_by_country.jsp?itemId=105&displayCurrency=EUR",
"Quality of Life": "https://www.numbeo.com/quality-of-life/in/city",
"Crime": "https://www.numbeo.com/crime/in/city",
"Pollution": "https://www.numbeo.com/pollution/in/city"}

Now we design the structure of the dataset we need, by creating new columns to each new element we are adding:

In [29]:
for element in url_dictionary: #creating a loop, as each new column corresponds to a part of Numbeo where we are taking info
    city_data.loc[:, element] = None #filling the new columns with none

After this, we create a function for each new column. Each function has the goal to read the HTML, finding the table where the info is and (as there are many indicators) we sum all the rows, storing it (these functions return that value):

In [ ]:
def food_prices(readable_html): 
    rows = readable_html.find_all("tr", class_=["tr_standard", "tr_highlighted"]) #finding in the HTML
    total = 0 #initiate the count of all indicators
    for row in rows: #iterate through each row/indicator
        tds = row.find_all("td") #find the number
        price = tds[1].get_text(strip=True).replace("\xa0", "").replace("(", "").replace(")", "") #series of lines treating the numbers as strings, to ensure right numbers are scrapped
        number = float(re.search(r"\d+[\.,]?\d*", price).group().replace(',', '.')) #2nd line, using regex to ensure we get the format we want - and need - for further analysis, allowing us to avoid the need of preprocessing
        total += number #summing the indicators 
    return float(total)

def gas_prices(readable_html):
    value = readable_html.find("span", class_="first_currency").text #target the specific span class that holds the price
    number = float(re.search(r"\d+[\.,]?\d*", value).group().replace(',', '.')) #extract numeric value via regex and normalize decimal separator
    return number

def quality_of_life(readable_html):
    total = 0 #initiate accumulator
    for element in readable_html.find_all("td", style="text-align: right"): #iterate through right-aligned table cells where data usually sits
        text = ''.join(digit for digit in element.get_text(strip=True) if digit.isdigit() or digit == '.') #filter string to keep only digits and dots (cleaner alternative to regex)
        if text: #ensure the string is not empty before converting
            total += float(text) #convert to float and add to total
    return total

def crime(readable_html):
    total = 0 #initiate accumulator
    for element in readable_html.find_all("td", style="text-align: right"): #target cells with right alignment
        text = ''.join(digit for digit in element.get_text(strip=True) if digit.isdigit() or digit == '.') #strip out all non-numeric characters except the decimal point
        if text: #check if valid number exists
            total += float(text) #sum the crime index value
    return total

def pollution(readable_html):
    total = 0 #initiate accumulator
    for element in readable_html.find_all("td", style="text-align: right"): #find all numerical data cells
        text = ''.join(digit for digit in element.get_text(strip=True) if digit.isdigit() or digit == '.') #cleaning the string to ensure only the raw number remains
        if text: #verify we have a number string
            total += float(text) #add pollution index to total
    return total

def salary(readable_html, country):
    #splits the entire body text by specific phrases to isolate the exact section containing salary info
    li = [x.strip() for x in readable_html.body.text.split("(After Tax)")[4].split("Last Update")[0].split("\n") if x.strip() != ""] 
    for item in li: #iterate through the isolated list of text lines
        if country in item: #match the specific country line
            number = float(re.search(r"\d+\.\d+", item).group()) #extract the specific salary float value
            return number #return immediately once the country is found

Now we finalise the webscrapping implementation with this function <code>get_info</code>, this will ensure the webscrapping is made, printing any errors (of not finding the right page) and ensure also that the names of the cities match the ones the Numbeo website uses:

In [ ]:
def get_info(data):
    for index in data.index: #iterate through each city in the dataframe
        city_name = data.loc[index, "City"]
        
        #checks to rename cities so they match the specific URL format in the numbeo website
        if city_name == "Bruges":
            city_name = "Brugge"
        elif city_name == "Lefkosia":
            city_name = "Nicosia"
        elif city_name == "Lemesos":
            city_name = "Limassol"
        elif city_name == "Frankfurt am Main":
            city_name = "Frankfurt"
        elif city_name == "Seville":
            city_name = "Sevilla"
        elif city_name == "The Hague":
            city_name = "The-Hague-Den-Haag-Netherlands"
        elif city_name == "Cracow":
            city_name = "Krakow-Cracow"
        
        country_name = data.loc[index, "Country"] #retrieve country name needed for salary calculation
        
        for char in url_dictionary: #loop through each category (key) in your URL dictionary
            url = url_dictionary[char].replace("city", city_name) #insert the corrected city name into the URL string
            html = requests.get(url) #send the request to the website
            readable_html = BeautifulSoup(html.text, "html.parser") #parse the returned HTML

            # Check which category we are currently processing and apply the specific scraping function
            if char == "Food Prices":
                try:
                    data.loc[index, char] = food_prices(readable_html) #attempt to scrape and assign food prices
                except:
                    data.loc[index, char] = 0 #if scraping fails, assign 0 to avoid breaking the loop

            elif char == "Gas Prices Calculator":
                try:
                    data.loc[index, char] = gas_prices(readable_html) #attempt to scrape gas prices
                except:
                    data.loc[index, char] = 0 #assign 0 on error

            elif char == "Salary Calculator":
                try:
                    data.loc[index, char] = salary(readable_html, country_name) #attempt to scrape salary (needs country argument)
                except:
                    data.loc[index, char] = 0 #assign 0 on error

            elif char == "Quality of Life":
                try:
                    data.loc[index, char] = quality_of_life(readable_html) #attempt to scrape quality of life index
                except:
                    data.loc[index, char] = 0 #assign 0 on error

            elif char == "Crime":
                try:
                    data.loc[index, char] = crime(readable_html) #attempt to scrape crime index
                except:
                    data.loc[index, char] = 0 #assign 0 on error

            elif char == "Pollution":
                try:
                    data.loc[index, char] = pollution(readable_html) #attempt to scrape pollution index
                except:
                    data.loc[index, char] = 0 #assign 0 on error

Running the <code>get_info()</code> function on the entire dataset, to complement it.

<b>NOTE</b>: We DO NOT RECOMMEND running this function, as the using of <code>Requests</code> will spike a block on the website and ask for the user manually complete a test to check the user is not a robot. We did this only once to retrive the information and then commented this line:

In [ ]:
#get_info(city_data)

Check the information we got:

In [45]:
city_data[['City', 'Country', 'Food Prices', 'Gas Prices Calculator', 'Salary Calculator', 'Quality of Life', 'Crime', 'Pollution']]

,City,Country,Food Prices,Gas Prices Calculator,Salary Calculator,Quality of Life,Crime,Pollution
0,Vienna,Austria,22.50,1.55,2639.33,907.47,626.08,874.30
1,Salzburg,Austria,23.90,1.50,2639.33,872.83,545.52,891.73
2,Brussels,Belgium,20.36,1.64,2624.98,817.91,922.65,1016.64
3,Antwerp,Belgium,18.85,1.63,2624.98,849.49,753.15,1000.14
4,Gent,Belgium,19.19,1.61,2624.98,913.83,584.52,900.27
...,...,...,...,...,...,...,...,...
79,Stockholm,Sweden,235.67,17.16,2815.29,803.24,814.14,871.41
80,Gothenburg,Sweden,216.21,17.43,2815.29,884.28,801.30,876.04
81,Malmo,Sweden,232.77,18.08,2815.29,827.19,905.06,862.94
82,Ankara,Turkiye,580.73,51.24,NaN,717.88,747.07,1062.28


In [35]:
url = "https://www.numbeo.com/cost-of-living/prices_by_country.jsp?itemId=105&displayCurrency=EUR"
html = requests.get(url)
readable_html = BeautifulSoup(html.text, "html.parser")
li = [x.strip() for x in readable_html.body.text.split("(After Tax)")[4].split("Last Update")[0].split("\n") if x.strip() != ""] 
for item in li:
    if "Portugal" in item:
        print(item)
        number = float(re.search(r"\d+\.\d+", item).group())
        print(number)

Portugal1124.41
1124.41


In [46]:
city_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 73 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Population Density                      84 non-null     float64
 1   Population                              84 non-null     int64  
 2   Working Age Population                  84 non-null     int64  
 3   Youth Dependency Ratio                  84 non-null     float64
 4   Unemployment Rate                       84 non-null     float64
 5   GDP per Capita                          84 non-null     float64
 6   Days of very strong heat stress         84 non-null     int64  
 7   Average Monthly Salary                  84 non-null     int64  
 8   Average Rent Price                      84 non-null     int64  
 9   Average Cost of Living                  84 non-null     int64  
 10  Average Price Groceries                 84 non-null     float64


Now we exported this, dataset with the webscrapping informations for personnal further exploration (we didn't use this file on our project, thus it is not in the delivered files):

In [ ]:
#pd.to_csv('city_data.csv', index = 'False')

<hr style = "border: 3px solid #7328baff;">
<h2 style = "color : #7328baff;">Dashboard</h2>
<p style = "font-size : 16px;">Here we designed the dashboard to complete both the comparison and the recommendation of city.</p>
<br>

Import of Images

In [36]:
country_flag_urls = {
    "Austria": "https://flagcdn.com/w160/at.png",
    "Belgium": "https://flagcdn.com/w160/be.png",
    "Bulgaria": "https://flagcdn.com/w160/bg.png",
    "Switzerland": "https://flagcdn.com/w160/ch.png",
    "Cyprus": "https://flagcdn.com/w160/cy.png",
    "Czechia": "https://flagcdn.com/w160/cz.png",
    "Germany": "https://flagcdn.com/w160/de.png",
    "Denmark": "https://flagcdn.com/w160/dk.png",
    "Spain": "https://flagcdn.com/w160/es.png",
    "Estonia": "https://flagcdn.com/w160/ee.png",
    "Finland": "https://flagcdn.com/w160/fi.png",
    "France": "https://flagcdn.com/w160/fr.png",
    "United Kingdom": "https://flagcdn.com/w160/gb.png",
    "Greece": "https://flagcdn.com/w160/gr.png",
    "Croatia": "https://flagcdn.com/w160/hr.png",
    "Hungary": "https://flagcdn.com/w160/hu.png",
    "Ireland": "https://flagcdn.com/w160/ie.png",
    "Italy": "https://flagcdn.com/w160/it.png",
    "Luxembourg": "https://flagcdn.com/w160/lu.png",
    "Latvia": "https://flagcdn.com/w160/lv.png",
    "Malta": "https://flagcdn.com/w160/mt.png",
    "Netherlands": "https://flagcdn.com/w160/nl.png",
    "Norway": "https://flagcdn.com/w160/no.png",
    "Poland": "https://flagcdn.com/w160/pl.png",
    "Portugal": "https://flagcdn.com/w160/pt.png",
    "Romania": "https://flagcdn.com/w160/ro.png",
    "Slovak Republic": "https://flagcdn.com/w160/sk.png",
    "Slovenia": "https://flagcdn.com/w160/si.png",
    "Sweden": "https://flagcdn.com/w160/se.png",
    "Turkiye": "https://flagcdn.com/w160/tr.png"
}

In [37]:
city_data["Image Url"] = None
for index in city_data.index:
    for country, url in country_flag_urls.items():
        if country in city_data.loc[index, "Country"]:
            city_data.loc[index, "Image Url"] = url
            break

In [38]:
#city_data.loc[36, "Country"]

Dashboard

In [39]:
cols_needed = ['Population Density', 'Population', 'Working Age Population',
       'Youth Dependency Ratio', 'Unemployment Rate', 'GDP per Capita',
       'Days of very strong heat stress', 'Average Monthly Salary',
       'Average Rent Price', 'Average Cost of Living',
       'Food Prices', 'Gas Prices Calculator', 'Salary Calculator',
       'Quality of Life', 'Crime', 'Pollution']

In [ ]:
# Initialize Panel extension
pn.extension('plotly')

# ======================================
# DATA PREPARATION
# ======================================
# Clean column names and fill NA values
city_data.columns = [col.strip() for col in city_data.columns]
city_data = city_data.fillna(0)

# ======================================
# STYLES
# ======================================
dashboard_style = {'background': '#F7F9FC', 'padding': '25px', 'border-radius': '15px'}
header_style = {'background': '#2A3B4D', 'padding': '5px 25px', 'border-radius': '10px', 'margin': '0 0 20px 0'}
card_style = {'background': '#FFFFFF', 'border': '1px solid #E0E0E0', 'border-radius': '10px', 'padding': '25px', 'margin': '10px 0'}
slider_style = {'background': '#FFFFFF', 'border-radius': '8px', 'padding': '15px', 'margin': '8px 0', 'box-shadow': '0 2px 12px rgba(0, 0, 0, 0.05)', 'border-left': '4px solid #0D6EFD',  'border-top': '1px solid #E0E0E0', 'border-right': '1px solid #E0E0E0', 'border-bottom': '1px solid #E0E0E0'}
infobox_style = {'background': '#FFFFFF', 'border-left': '5px solid #0D6EFD', 'border-radius': '8px', 'padding': '10px', 'margin': '8px', 'box-shadow': '0 2px 6px 0 rgba(0,0,0,0.07)', 'width': '260px', 'height': '80px', 'box-sizing': 'border-box', 'text-align': 'center', 'display': 'flex', 'flex-direction': 'column', 'justify-content': 'center'}

# ======================================
# WIDGETS
# ======================================
# City selectors
country_1 = pn.widgets.Select( name="City", options=sorted(list(city_data["City"])), min_width=200, width_policy='max', sizing_mode='stretch_width')
country_2 = pn.widgets.Select(name="City", options=list(city_data["City"]), min_width=200, width_policy='max', sizing_mode='stretch_width')

# View selector
select = pn.widgets.RadioButtonGroup( name="Select", options=["Characteristics", "Compare Cities"],  button_type='primary', button_style='outline', min_width=200, sizing_mode='stretch_width')

# Navigation buttons
big_button = pn.widgets.Button(name="Your Country Is...", button_type='primary', height=40, styles={ 'font-size': '16px', 'font-weight': 'bold', 'padding': '5px'}, sizing_mode='stretch_width', width_policy='max')
go_back_button = pn.widgets.Button( name='Go Back', button_type='default', sizing_mode='stretch_width')

# ======================================
# DATA PROCESSING FUNCTIONS
# ======================================
def scale(data, cols):
    """Scale numeric columns for comparison"""
    for col in cols:
        data[f"{col} Scaled"] = data[col] / data[col].sum()

# Apply scaling
scale(city_data, cols_needed)

def best_country(sliders, data):
    """Calculate best matching country based on slider weights"""
    final = pd.Series(0, index=data["City"], name="Scores")
    for index in final.index:
        total = 0
        for name, slider in sliders.items():
            col = f"{name} Scaled"
            total += slider.value * data.loc[data["City"] == index, col].iloc[0]
        final[index] = total
    final = final.sort_values(ascending=False)
    return final.idxmax(), final

def update_display(*_):
    """Update display when sliders change"""
    best, final = best_country(sliders, city_data)
    return best, final

# ======================================
# UI COMPONENTS
# ======================================
def binary(option):
    """Render either single city or comparison view based on selection"""
    def get_image(city):
        return pn.pane.PNG(city_data[city_data["City"] == city].iloc[0]["Image Url"], width=240, height=160)

    interative_image = pn.bind(get_image, country_1)  
    
    if option == "Characteristics":
        return pn.Column(pn.pane.Markdown("### Your most wanted country"), interative_image, styles=card_style)
    else:
        return pn.Column(
            pn.pane.Markdown("### Choose your country"), 
            pn.Column(country_1, country_2), 
            styles=card_style
        )

def chars_left(selected_chars, city):
    """Display selected characteristics for a country"""
    global boards
    boards = []
    
    emoji_map = {
    "Population Density": "👨‍👩‍👧",
    "Population": "🌎",
    "Working Age Population": "💼🔞",
    "Youth Dependency Ratio": "👶🏻",
    "Unemployment Rate": "💼📉",
    "GDP per Capita": "💰",
    "Days of very strong heat stress": "🌡️🔥",
    "Average Monthly Salary": "💼💰",
    "Average Rent Price": "🏠💲",
    "Average Cost of Living": "💸👨‍👨‍👧‍👦",
    "Food Prices": "🥦💲",
    "Gas Prices Calculator": "⛽💲",
    "Salary Calculator": "📊💼",
    "Quality of Life": "🌿😊",
    "Crime": "🚔⚠️",
    "Pollution": "🏭😷"
}

    
    for char in selected_chars:
        if char in emoji_map:
            emoji = emoji_map[char]
            board = pn.pane.Markdown(f"<div style='font-size:8pt'><b>{char}</b> {emoji}</div><br><div style='font-size:7pt'>Current value: <b>{round(city_data[city_data["City"] == city].iloc[0][char], 0)}</b></div>", styles=infobox_style)
            boards.append(board)

    # Create rows of 3 columns
    rows = []
    for i in range(0, len(boards), 3):
        rows.append(pn.Row(*boards[i:i+3]))
    
    return pn.Column(*rows)

def all_chars(chars, city_1, city_2):
    """Display comparison of all characteristics between two countries"""
    global boards
    boards = []
    
    emoji_map = {
    "Population Density": "👨‍👩‍👧",
    "Population": "🌎",
    "Working Age Population": "💼🔞",
    "Youth Dependency Ratio": "👶🏻",
    "Unemployment Rate": "💼📉",
    "GDP per Capita": "💰",
    "Days of very strong heat stress": "🌡️🔥",
    "Average Monthly Salary": "💼💰",
    "Average Rent Price": "🏠💲",
    "Average Cost of Living": "💸👨‍👨‍👧‍👦",
    "Food Prices": "🥦💲",
    "Gas Prices Calculator": "⛽💲",
    "Salary Calculator": "📊💼",
    "Quality of Life": "🌿😊",
    "Crime": "🚔⚠️",
    "Pollution": "🏭😷"
}

    
    # Get image URLs
    img1_url = city_data[city_data["City"] == city_1].iloc[0]["Image Url"]
    img2_url = city_data[city_data["City"] == city_2].iloc[0]["Image Url"]
    
    for char in chars:
        if char in emoji_map:
            # Create image panes with proper HTML
            image1 = pn.pane.HTML(f'<img src="{img1_url}" width="30" height="20" style="vertical-align:middle">')
            image2 = pn.pane.HTML(f'<img src="{img2_url}" width="30" height="20" style="vertical-align:middle">')
            emoji = emoji_map[char]
            
            comparison_row = pn.Row(
                image1,
                pn.pane.HTML(f"<div style='text-align:center; font-size:9px'><b>{round(city_data[city_data["City"] == city_1].iloc[0][char], 0)}</b></div>"),
                pn.pane.HTML("<div style='text-align:center; font-size:10px'><b>vs</b></div>"),
                pn.pane.HTML(f"<div style='text-align:center; font-size:9px'><b>{round(city_data[city_data["City"] == city_2].iloc[0][char], 0)}</b></div>"),
                image2, align='center')
        
            board = pn.Column(pn.Column(pn.pane.Markdown(f"<div style='text-align:center; font-size:12px'><b>{char} {emoji}</b></div>"), align='center'), comparison_row, styles=infobox_style, align='center')
            
            boards.append(board)

    # Create rows of 3 columns
    rows = []
    for i in range(0, len(boards), 3):
        rows.append(pn.Row(*boards[i:i+3]))
    
    return pn.Column(*rows)

def create_pie_chart(*values):
    """Create a pie chart showing weight distribution"""
    all_labels = list(sliders.keys())
    values_list = []
    labels_list = []
    
    for i in range(len(values)):
        if values[i] > 0:
            values_list.append(values[i])
            labels_list.append(all_labels[i])
    
    if not values_list:
        return pn.pane.Markdown("### Adjust the sliders to see the weight distribution")

    n = len(values_list)
    blue_colors = [f'rgb({int(30 + i * (225/n))}, {int(100 + i * (155/n))}, {int(180 + i * (75/n))})' for i in range(n)]
    
    pie = go.Pie(labels=labels_list, values=values_list, textinfo='label+percent', hole=0.3, textfont_size=12, textposition='inside', marker_colors=blue_colors)
    
    fig = go.Figure(data=[pie])
    
    fig.update_layout(margin=dict(l=10, r=10, t=30, b=10), showlegend=False, height=250, width=400, xaxis=dict(showgrid=False, zeroline=False), yaxis=dict(showgrid=False, zeroline=False))
    
    return pn.pane.Plotly(fig, config={'displayModeBar': False}, width=400, height=250, sizing_mode='fixed')

# ======================================
# SLIDERS CREATION
# ======================================
sliders = {}
for col in cols_needed:
    sliders[col] = pn.widgets.IntSlider(name=col, start=0, end=10, step=1, width=200, styles=slider_style, width_policy='max')

# Create 3 columns for sliders
slider_columns = [[], [], []]
for i, (col, slider) in enumerate(sliders.items()):
    column_index = i % 3
    slider_columns[column_index].append(slider)

# Create Rows for each column of sliders
slider_rows = []
for col_sliders in slider_columns:
    if col_sliders:
        slider_rows.append(pn.Column(*col_sliders, width=190))

# Create sliders panel
sliders_panel = pn.Column(pn.pane.Markdown("### On a scale from one to ten, how important are these characteristics?", styles={'color': '#0D6EFD'}), pn.Row(*slider_rows), styles={'background': '#FFFFFF', 'padding': '15px', 'border-radius': '8px'}, width_policy='max')

# Initialize slider values
slider_values = [slider.param.value for slider in sliders.values()]

# Create the pie chart panel
pie_chart = pn.panel(pn.bind(create_pie_chart, *slider_values), config={'displayModeBar': False})

# ======================================
# LAYOUTS
# ======================================
header = pn.Row(pn.pane.Markdown("## Dashboard", styles={'color': '#FFFFFF'}), styles=header_style, sizing_mode='stretch_width')

def one_country():
    """Layout for single country view"""
    header = pn.Row(pn.pane.Markdown("## Dashboard", styles={'color': '#FFFFFF'}), styles=header_style, sizing_mode='stretch_width')
    under_header = pn.Row(select, styles=card_style, width=413)
    left = pn.Column(pn.panel(interactive_binary), styles=card_style, width=373, sizing_mode='fixed')
    return pn.Column( header, pn.Row(pn.Column(under_header, pie_chart), pn.Spacer(width = 150) , sliders_panel), pn.Row(big_button, styles={'background': '#2A3B4D', 'padding': '5px 25px', 'border-radius': '10px', 'margin': '0 0 10px 0'}, sizing_mode='stretch_width'))

def two_countries():
    """Layout for comparing two countries"""
    header = pn.Row(pn.pane.Markdown("## Dashboard", styles={'color': '#FFFFFF'}), styles=header_style, sizing_mode='stretch_width')
    under_header = pn.Row(select, styles=card_style)
    return pn.Column(header, pn.Row(pn.Column(under_header, interactive_binary, sizing_mode='fixed'), pn.Spacer(width = 75) ,interactive_boards_all, styles=card_style))

def layout_choice(selection):
    """Choose between single country or comparison layout"""
    if selection == "Characteristics":
        return one_country()
    else:
        return two_countries()

# ======================================
# INTERACTIVE BINDINGS
# ======================================
# Interaction between the select "Characteristics" and "Comparison of the two cities"
interactive_binary = pn.bind(binary, select)

# Comparison of the layout choice function and the select box
interactive_layout = pn.bind(layout_choice, select)

# Get all characteristics for comparison
chars = cols_needed

# Bind the all_chars function to city selectors
interactive_boards_all = pn.bind(lambda city1, city2: all_chars(chars, city1, city2), country_1.param.value, country_2.param.value)

# ======================================
# PAGES
# ======================================
def page1():
    """Results page showing best matching country"""
    
    best, scores = best_country(sliders, city_data)
    #print(best, scores)
    best_score = scores[best]
    #print(best_score)

    chars = cols_needed

    results = chars_left(chars, best)
    #print(results)
    def get_image(_):
        return pn.pane.PNG(city_data[city_data["City"] == best].iloc[0]["Image Url"], width=240, height=160)
    
    return pn.Column(header,pn.Row(pn.Column( pn.pane.Markdown("## Your Perfect Match"), pn.bind(get_image, None), pn.pane.Markdown(f"### {best}"), results), pn.Spacer(width = 100) ,pn.Column(pn.pane.Markdown(f"**Match Score:** {best_score:.2f}"), pn.pane.Markdown("### All Scores:"), pn.pane.DataFrame(scores.to_frame("Score").head(26)))), go_back_button, margin=20)

# ======================================
# NAVIGATION
# ======================================
def show_page(page_name, event=None):
    """Handle page navigation"""
    dashboard.clear()
    
    if page_name == 'home':
        # Always check the current select value
        selection = select.value
        if selection == "Characteristics":
            page = one_country()
        else:
            page = two_countries()
        big_button.on_click(lambda e: show_page('page1'))
        dashboard.append(page)
        
    elif page_name == 'page1':
        page = page1()
        go_back_button.on_click(lambda e: show_page('home'))
        dashboard.append(page)

# ======================================
# INITIALIZATION
# ======================================
# Create dashboard
dashboard = pn.Column(styles=dashboard_style, sizing_mode='stretch_width')

# Set up view selector callback
def on_select_change(event):
    show_page('home')

select.param.watch(on_select_change, 'value')

# Bind the best country display
best_country_display = pn.bind(update_display, *[s.param.value for s in sliders.values()])

# Start with home page
show_page('home')

# ======================================
# SERVE THE DASHBOARD
# ======================================
pn.serve(dashboard)

Launching server at http://localhost:54231


<hr style = "border: 3px solid #7328baff;">
<h2 style = "color : #7328baff;">Map of Holiday Destinations</h2>
<p style = "font-size : 16px;">Here we used <code>geopandas</code> to implement a map where you can see the destinations around your chosen city.</p>
<br>

For this, we got a dataset from Kaggle (<code>https://www.kaggle.com/datasets/faizadani/european-tour-destinations-dataset</code>) and we added to the Project Files as 'destinations_csv'

In [ ]:
city_data = gpd.read_file('geo_city_data.csv')

city_data = gpd.GeoDataFrame(
    city_data,
    geometry=gpd.points_from_xy(city_data['Interactive Longitude'], city_data['Interactive Latitude']),
    crs="EPSG:4326"  # Defines the coordinate system as WGS
)

city_data = city_data[['City', 'Country', 'geometry']]

city_data.head()

In [ ]:
destination_data = gpd.read_file('destinations.csv')

destination_data = gpd.GeoDataFrame(
    destination_data, 
    geometry=gpd.points_from_xy(destination_data['Longitude'], destination_data['Latitude']),
    crs="EPSG:4326"  # Defines the coordinate system as WGS84 (standard lat/lon)
)

destination_data ['City'] = destination_data['Destination']

In [ ]:
destination_data = destination_data[['City', 'Country', 'geometry', 'Approximate Annual Tourists', 'Currency', 'Language', 'Famous Foods', 'Best Time to Visit', 'Safety', 'Cost of Living', 'Cultural Significance', 'Description']]
destination_data.head()

In [ ]:
distance = 1000  # KM YOU ARE WILLING TO GO TRAVEL ON HOLIDAYS
city_choosen = 'Athens'

In [ ]:
# 1. Ensure both datasets are in the Metric system (Meters)
# We perform this explicitly to guarantee 'my_city' is in meters
cities_metric = city_data.to_crs(epsg=3857)
dest_metric = destination_data.to_crs(epsg=3857)



# We use the metric dataframe we just created
my_city_geom = cities_metric[cities_metric['City'] == city_choosen].iloc[0].geometry

# 1. Extract the row for your chosen city
city_row = cities_metric[cities_metric['City'] == city_choosen].copy()

# 2. Rename 'City' to 'Destination' to match the column names in destination_data
# This ensures the tooltip shows "Zagreb" correctly
city_row = city_row.rename(columns={'City': 'Destination'})

# 3. Add a category so you can distinguish it on the map (Optional)
city_row['Category'] = 'My Location'

# 4. Add it to the main destinations dataframe
dest_metric = pd.concat([dest_metric, city_row], ignore_index=True)

# 3. Create a 50km buffer (50,000 meters)
# Since my_city_geom is in meters, 50000 = 50km
search_area = my_city_geom.buffer(distance * 100) 

# 4. Filter destinations within that search area
nearby_destinations = dest_metric[dest_metric.geometry.within(search_area)]

print(f"Found {len(nearby_destinations)} destinations within {distance}km of {city_choosen}.")

# 5. Plot the results
# We use the 'nearby_destinations' as the base map. 
# If it's empty, we create a map from the search_area instead to avoid errors.
if len(nearby_destinations) > 0:
    m = nearby_destinations.explore(color='red', marker_type='marker', popup=True)
    # Add the search circle in blue
    gpd.GeoSeries([search_area], crs=3857).explore(m=m, color='blue', alpha=0.3)
else:
    # If no destinations found, just show the blue search circle
    m = gpd.GeoSeries([search_area], crs=3857).explore(color='blue', alpha=0.3)

m = nearby_destinations.explore(
    color='red',
    marker_type='marker',
    tiles='CartoDB positron' 
)

# Add the blue search circle to the same map
gpd.GeoSeries([search_area], crs=3857).explore(m=m, color='blue', alpha=0.3)
m